In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from itertools import combinations
from scipy.stats import chisquare, entropy, ks_2samp, wasserstein_distance
from sklearn.cluster import DBSCAN, KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import umap.umap_ as umap


In [ ]:
real_path = "../../data/data_for_CTGAN/CTGAN_basedata.csv"
syn_path = "../../data/synthetic_result_data/536_synthetic_train_data.csv"
df_real = pd.read_csv(real_path)
df_syn = pd.read_csv(syn_path)

# Statistical Analysis - Statistical Similarity Score

In [ ]:
def compute_subscores(original: pd.DataFrame, synthetic: pd.DataFrame, alpha_kl=1.0, mmd_gamma=1.0, weights=None):
    """
    Compute Statistical Similarity Score (SSS) subscores for each feature and overall score.
    Returns a dict with per-feature subscores and the overall SSS.
    """
    def compute_mmd(x, y, gamma):
        """Compute unbiased MMD with RBF kernel, returning a scalar."""
        X = x.values.flatten()
        Y = y.values.flatten()
        Kxx = np.exp(-gamma * (X[:, None] - X[None, :])**2)
        Kyy = np.exp(-gamma * (Y[:, None] - Y[None, :])**2)
        Kxy = np.exp(-gamma * (X[:, None] - Y[None, :])**2)
        n, m = len(X), len(Y)
        mmd2 = (
            (Kxx.sum() - np.trace(Kxx)) / (n * (n - 1))
            + (Kyy.sum() - np.trace(Kyy)) / (m * (m - 1))
            - 2 * Kxy.sum() / (n * m)
        )
        return np.sqrt(max(mmd2, 0))

    # Sprechende Metriknamen
    metric_names = ['mean','median','variance','ks','chi2','wasserstein','mmd','kl','coverage']

    if weights is None:
        # Default: gleichmäßige Verteilung
        weights = {name: 1/len(metric_names) for name in metric_names}

    results = {}
    feature_scores = []

    for col in original.columns:
        o = original[col]
        s = synthetic[col]
        subscores = {}

        if pd.api.types.is_numeric_dtype(o):
            R = o.max() - o.min()
            delta_mean = abs(o.mean() - s.mean())
            subscores['mean'] = max(0, 1 - delta_mean / R) if R > 0 else 1.0

            delta_median = abs(o.median() - s.median())
            subscores['median'] = max(0, 1 - delta_median / R) if R > 0 else 1.0

            V = max(o.var(), s.var())
            delta_var = abs(o.var() - s.var())
            subscores['variance'] = max(0, 1 - delta_var / V) if V > 0 else 1.0

            subscores['ks'] = ks_2samp(o, s).pvalue

            w_dist = wasserstein_distance(o, s)
            subscores['wasserstein'] = 1 / (1 + w_dist)

            mmd_val = compute_mmd(o, s, mmd_gamma)
            subscores['mmd'] = 1 / (1 + mmd_val)

            bins = min(50, max(o.nunique(), s.nunique()))
            p_hist, _ = np.histogram(o, bins=bins, density=True)
            q_hist, _ = np.histogram(s, bins=bins, density=True)
            p_hist += 1e-8
            q_hist += 1e-8
            kl_div = entropy(p_hist, q_hist)
            subscores['kl'] = np.exp(-alpha_kl * kl_div)

            cov = len(np.intersect1d(o.unique(), s.unique())) / o.nunique() if o.nunique() > 0 else 1.0
            subscores['coverage'] = cov

        else:
            real_counts = o.value_counts().sort_index()
            syn_counts = s.value_counts().reindex(real_counts.index, fill_value=0)
            stat, p_value = chisquare(f_obs=syn_counts, f_exp=real_counts)
            subscores['chi2'] = p_value
            cov = len(set(o.unique()) & set(s.unique())) / len(o.unique()) if o.nunique() > 0 else 1.0
            subscores['coverage'] = cov

        # Nur vorhandene Metriken werten
        available_metrics = [k for k in subscores if k in weights]
        total_weight = sum(weights[k] for k in available_metrics)

        score = sum(weights[k] * subscores[k] for k in available_metrics) / total_weight if total_weight > 0 else 0
        results[col] = {'subscores': subscores, 'feature_score': score}
        feature_scores.append(score)

    overall_sss = float(np.mean(feature_scores))
    return {'per_feature': results, 'SSS': overall_sss}


In [ ]:
custom_weights = {
    'mean': 0.05,
    'median': 0.05,
    'variance': 0.05,
    'ks': 0.1,
    'chi2': 0.1,
    'wasserstein': 0.2,
    'mmd': 0.2,
    'kl': 0.2,
    'coverage': 0.05
}


result = compute_subscores(df_real, df_syn, weights=custom_weights)
print("Statistical Similarity Score (SSS):", result['SSS'])


for feature, data in result['per_feature'].items():
    print(f"\nFeature: {feature}")
    print(" Subscores:", data['subscores'])
    print(" Feature score:", data['feature_score'])


# Structural Consistency - Structural Consistency Score

In [ ]:
def compute_structural_similarity_score(original: pd.DataFrame,
                                        synthetic: pd.DataFrame,
                                        eps=0.5,
                                        min_samples=5,
                                        weights=None):
    """
    Computes a universal structural similarity score ∈ [0,1], where 1 is best.
    Integrates:
      - Cluster Purity (KMeans & DBSCAN)
      - Adjusted Rand Index (KMeans & DBSCAN)
      - Cosine Similarity of global mean feature vectors
    Each subscore ∈ [0,1]; final score is the average.
    """
    # Preprocessing: one-hot encode categorical + standardize numeric
    num_cols = original.select_dtypes(include=[np.number]).columns.intersection(
               synthetic.select_dtypes(include=[np.number]).columns)
    cat_cols = original.select_dtypes(exclude=[np.number]).columns.intersection(
               synthetic.select_dtypes(exclude=[np.number]).columns)
    
    if len(cat_cols) > 0:
        combined_cat = pd.concat([original[cat_cols], synthetic[cat_cols]], axis=0)
        encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        encoder.fit(combined_cat)
        orig_cat = encoder.transform(original[cat_cols])
        syn_cat = encoder.transform(synthetic[cat_cols])
    else:
        orig_cat = np.empty((len(original), 0))
        syn_cat = np.empty((len(synthetic), 0))
    
    orig_num = original[num_cols].to_numpy()
    syn_num  = synthetic[num_cols].to_numpy()
    scaler = StandardScaler().fit(np.vstack([orig_num, syn_num]))
    orig_num = scaler.transform(orig_num)
    syn_num  = scaler.transform(syn_num)
    
    # Feature matrices
    X_orig = np.hstack([orig_num, orig_cat])
    X_syn  = np.hstack([syn_num,  syn_cat])
    labels = np.concatenate([np.zeros(len(X_orig)), np.ones(len(X_syn))])
    X_joint = np.vstack([X_orig, X_syn])
    
    # 1. Cluster Purity Subscores
    def purity_subscore(cl_labels):
        subs = []
        for c in np.unique(cl_labels):
            mask = (cl_labels == c)
            counts = np.bincount(labels[mask].astype(int), minlength=2)
            if counts.sum() == 0: continue
            purity = counts.max() / counts.sum()
            # Map purity 0.5->1.0, purity 1.0->0.0
            subs.append(max(0, 1 - abs(purity - 0.5) / 0.5))
        return np.mean(subs) if subs else 0.0
    
    km_labels = KMeans(n_clusters=2, random_state=42).fit_predict(X_joint)
    db_labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_joint)
    s_purity_km = purity_subscore(km_labels)
    s_purity_db = purity_subscore(db_labels)
    
    # 2. ARI Subscores
    s_ari_km = 1 - abs(adjusted_rand_score(labels, km_labels))
    s_ari_db = 1 - abs(adjusted_rand_score(labels, db_labels))
    
    # 3. Cosine Similarity Subscore
    vec_orig = X_orig.mean(axis=0).reshape(1, -1)
    vec_syn  = X_syn.mean(axis=0).reshape(1, -1)
    cos_val = cosine_similarity(vec_orig, vec_syn)[0, 0]
    s_cos = (cos_val + 1) / 2  
    
    # Default equal weights across 5 subscores
    if weights is None:
        weights = {
            'purity_km': 0.15,
            'purity_db': 0.15,
            'ARI_km':    0.15,
            'ARI_db':    0.15,
            'cosine':    0.40
        } 
    
    # Combine
    structural_score = (
        weights['purity_km'] * s_purity_km +
        weights['purity_db'] * s_purity_db +
        weights['ARI_km']    * s_ari_km +
        weights['ARI_db']    * s_ari_db +
        weights['cosine']    * s_cos
    )
    
    return round(float(structural_score), 4)



In [ ]:

sim_score = compute_structural_similarity_score(df_real, df_syn)
print("Structure Consistency Score:", sim_score)